# Notebook 1 — Prithvi Patch Similarity (IBM-NASA, MAE-pretrained)

### *Hands-On Session: Earth Embeddings for EO — Retrieval, Discovery, and Change-Oriented Search*

**Instructors:** Fuxun Yu, Rishi Madhok (TerraByte AI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iceysteel/ESA-NASA-Workshop-2026/blob/main/Day%202/Track%202/Earth-Embeddings-EO/notebook_1_prithvi_patch_similarity.ipynb)

---

## What we're doing

In **Notebook 0** we saw that a *supervised* classifier produces useful embeddings at the **scene level**. Most operational EO workflows need something finer — *which pixels in this scene look similar?* — and we usually don't have labels.

Enter the **MAE (Masked Autoencoder)** pretraining recipe: hide 75% of input patches, ask a Vision Transformer to fill them back in, and the encoder learns a strong **per-patch** representation in the process — *no labels required*. [Prithvi-EO-2.0](https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M) is IBM Research and NASA's MAE-pretrained ViT for Harmonized Landsat–Sentinel-2 (HLS) imagery: **6 spectral bands × multi-temporal × 300M params**.

In this notebook we will:
1. Load the Prithvi-EO-2.0-300M **encoder** (no decoder, no fine-tuning).
2. Run it on a real HLS time-series chip over Mexico (4 timestamps, 6 bands).
3. Extract **per-patch embeddings** (14×14 grid in space, averaged over time).
4. Pick a **query patch** and compute cosine-similarity heatmaps to every other patch.

Notebook 2 will repeat this on the **same scene** with Meta's DINOv3 (RGB-only, self-distillation), so you can compare what each pretraining recipe considers 'similar'.


## Runtime

**Recommended:** `Runtime → Change runtime type → T4 GPU` (free) — ~30 s end-to-end. 
**Fallback:** CPU — ~1–2 min. Prithvi-EO-2.0-300M is a ViT-L; single forward pass on one chip is cheap.


In [ ]:
!pip install -q einops tifffile


In [ ]:
import json, importlib.util, time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from huggingface_hub import hf_hub_download

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)


## 1. Pull Prithvi-EO-2.0-300M from the HF Hub

The model is published as **raw weights + a small `prithvi_mae.py` file** containing the architecture, rather than a `transformers`-native checkpoint. We download three things:

- `prithvi_mae.py` — defines the `PrithviViT` encoder class.
- `config.json` — architecture hyperparameters + per-band normalization statistics.
- `Prithvi_EO_V2_300M.pt` — the pretrained weights (~1.2 GB).

Plus four bundled example **HLS GeoTIFF chips** over Mexico (4 timestamps, 6 spectral bands each — B02–B07, i.e. Blue, Green, Red, NIR, SWIR1, SWIR2).


In [ ]:
REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M'

t0 = time.time()
src_py  = hf_hub_download(REPO, 'prithvi_mae.py')
cfg_p   = hf_hub_download(REPO, 'config.json')
ckpt_p  = hf_hub_download(REPO, 'Prithvi_EO_V2_300M.pt')
tif_ids = [
    '2018026T173609',  # Jan 26
    '2018106T172859',  # Apr 16
    '2018201T172901',  # Jul 20
    '2018266T173029',  # Sep 23
]
tif_paths = [hf_hub_download(REPO, f'examples/Mexico_HLS.S30.T13REM.{i}.v2.0_cropped.tif') for i in tif_ids]
print(f'Downloads done in {time.time()-t0:.1f}s.')
print(f'Got {len(tif_paths)} HLS timestamps over Mexico.')


## 2. Instantiate the encoder and load weights

`prithvi_mae.py` is loaded as a Python module on the fly. We build **`PrithviViT`** — the encoder half of the MAE; we don't need the decoder for embeddings. The pretrained checkpoint contains both encoder and decoder weights under `encoder.*` / `decoder.*` prefixes, so we filter to encoder-only.


In [ ]:
spec = importlib.util.spec_from_file_location('prithvi_mae', src_py)
prithvi_mae = importlib.util.module_from_spec(spec); spec.loader.exec_module(prithvi_mae)

cfg = json.load(open(cfg_p))['pretrained_cfg']
init_keys = ['img_size','num_frames','patch_size','in_chans','embed_dim','depth',
             'num_heads','mlp_ratio','coords_encoding','coords_scale_learn']
encoder = prithvi_mae.PrithviViT(**{k: cfg[k] for k in init_keys})

sd_full = torch.load(ckpt_p, map_location='cpu', weights_only=False)
enc_sd  = {k[len('encoder.'):]: v for k, v in sd_full.items() if k.startswith('encoder.')}
missing, unexpected = encoder.load_state_dict(enc_sd, strict=True)
assert not missing and not unexpected, (missing, unexpected)

encoder = encoder.eval().to(device)
n_params = sum(p.numel() for p in encoder.parameters()) / 1e6
print(f'PrithviViT encoder loaded — {n_params:.1f}M params, embed_dim={cfg["embed_dim"]}, '
      f'patch_size={cfg["patch_size"]}, bands={cfg["bands"]}, frames={cfg["num_frames"]}')


## 3. Load and preprocess the HLS time-series

Each GeoTIFF is `(H, W, 6)` int16 surface reflectance. We:

1. Stack the 4 timestamps into one tensor `(T=4, H, W, C=6)`.
2. Center-crop to **224×224** — the model's native spatial size.
3. Reorder to `(B=1, C=6, T=4, H=224, W=224)` and **normalize** with the per-band mean/std from `config.json` (these are the *reflectance-scale* statistics Prithvi was trained with — *not* ImageNet stats).


In [ ]:
raw = np.stack([tifffile.imread(p) for p in tif_paths])           # (T,H,W,C)
print('Raw stack shape:', raw.shape, 'dtype:', raw.dtype,
      'range:', int(raw.min()), '..', int(raw.max()))

S = 224
H, W = raw.shape[1], raw.shape[2]
h0, w0 = (H - S) // 2, (W - S) // 2
chip = raw[:, h0:h0+S, w0:w0+S, :]                                # (T,S,S,C)

x = torch.from_numpy(chip).float().permute(3, 0, 1, 2).unsqueeze(0)  # (1,C,T,S,S)
mean = torch.tensor(cfg['mean']).view(1, 6, 1, 1, 1)
std  = torch.tensor(cfg['std']).view(1, 6, 1, 1, 1)
x = (x - mean) / std
print('Model input tensor:', tuple(x.shape))


### Visualise the input — RGB rendering of one timestamp

Prithvi bands are `[B02, B03, B04, B05, B06, B07]` = `[Blue, Green, Red, NIR, SWIR1, SWIR2]`. RGB = `(B04, B03, B02)`.


In [ ]:
def to_rgb(chip_thwc, t=2, p_low=2, p_high=98):
    """Render one timestamp as a contrast-stretched RGB uint8 image."""
    rgb = chip_thwc[t][..., [2, 1, 0]].astype(np.float32)          # R,G,B
    lo, hi = np.percentile(rgb, [p_low, p_high])
    rgb = np.clip((rgb - lo) / max(hi - lo, 1e-6), 0, 1)
    return rgb

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
labels = ['Jan 2018', 'Apr 2018', 'Jul 2018', 'Sep 2018']
for ax, t, lab in zip(axes, range(4), labels):
    ax.imshow(to_rgb(chip, t=t)); ax.set_title(lab); ax.axis('off')
plt.suptitle('Mexico HLS chip — 4 timestamps (R=B04, G=B03, B=B02)', y=1.02)
plt.tight_layout(); plt.show()


## 4. Extract per-patch embeddings

`PrithviViT.forward_features` returns **all 24 ViT block outputs**. We take the last layer.

Token layout: `[CLS] [t0_p0 t0_p1 … t0_p195] [t1_p0 …] [t2_p0 …] [t3_p0 …]` — that's `1 + 4 × 196 = 785` tokens of dim 1024.

We drop the CLS token, reshape to `(T=4, 14, 14, 1024)`, and **average over the temporal axis** to get a single `(14, 14, 1024)` spatial feature map per chip.


In [ ]:
with torch.inference_mode():
    t0 = time.time()
    feats_all_layers = encoder.forward_features(x.to(device))
    print(f'Forward pass: {time.time()-t0:.2f}s on {device}')

tokens = feats_all_layers[-1][0]                       # (785, 1024)
patches = tokens[1:].reshape(cfg['num_frames'], 14, 14, cfg['embed_dim'])  # (T,14,14,D)
spatial = patches.mean(dim=0).cpu().numpy()            # (14, 14, 1024)
print('Per-patch spatial feature map:', spatial.shape)


## 5. Patch-similarity heatmap

Pick a **query patch** — we use a coordinate in the bottom-left where there's a distinctive feature, but you can change `QUERY_RC` to anything in `[0..13] × [0..13]`. We:

1. L2-normalize all 196 patch vectors.
2. Take cosine similarity between the query patch and every other patch → 14×14 similarity grid.
3. Upsample to 224×224 and overlay on the RGB rendering.

Where the heatmap glows, Prithvi thinks those locations are **semantically similar** to the query — in HLS-feature terms, that's similar reflectance signatures *across* time and *across* spectral bands.


In [ ]:
QUERY_RC = (6, 6)   # (row, col) in the 14x14 patch grid
VIS_T    = 2        # which timestamp to render as the RGB background (0..3)

feat = torch.from_numpy(spatial).reshape(-1, spatial.shape[-1])        # (196, D)
feat = F.normalize(feat, dim=-1)
q = feat[QUERY_RC[0] * 14 + QUERY_RC[1]]
sim = (feat @ q).reshape(14, 14).numpy()                                # (14,14)
print(f'Similarity range: {sim.min():.3f} .. {sim.max():.3f}')

# Upsample 14x14 -> 224x224 for overlay
sim_up = F.interpolate(
    torch.from_numpy(sim)[None, None].float(),
    size=(S, S), mode='bilinear', align_corners=False
).squeeze().numpy()

rgb = to_rgb(chip, t=VIS_T)
qr, qc = QUERY_RC
qy, qx = qr * 16, qc * 16   # patch (r,c) -> pixel top-left

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb); axes[0].set_title('RGB')
axes[0].add_patch(Rectangle((qx, qy), 16, 16, ec='cyan', fc='none', lw=2.5))
axes[1].imshow(sim, cmap='magma'); axes[1].set_title(f'Raw 14x14 cosine sim (query={QUERY_RC})')
axes[1].add_patch(Rectangle((qc - 0.5, qr - 0.5), 1, 1, ec='cyan', fc='none', lw=2))
axes[2].imshow(rgb); axes[2].imshow(sim_up, cmap='magma', alpha=0.55)
axes[2].add_patch(Rectangle((qx, qy), 16, 16, ec='cyan', fc='none', lw=2.5))
axes[2].set_title('RGB + similarity overlay')
for ax in axes: ax.axis('off')
plt.suptitle('Prithvi-EO-2.0 patch similarity (cosine, last encoder layer)', y=1.02)
plt.tight_layout(); plt.show()


### Try a few different query patches at once

In [ ]:
queries = [(2, 2), (6, 6), (10, 10), (3, 11)]
rgb = to_rgb(chip, t=VIS_T)

fig, axes = plt.subplots(1, len(queries) + 1, figsize=(4.2 * (len(queries) + 1), 4.2))
axes[0].imshow(rgb); axes[0].set_title('RGB (Apr 2018)')
for (qr, qc), color in zip(queries, ['cyan', 'lime', 'orange', 'magenta']):
    axes[0].add_patch(Rectangle((qc * 16, qr * 16), 16, 16, ec=color, fc='none', lw=2))
axes[0].axis('off')

for ax, (qr, qc), color in zip(axes[1:], queries, ['cyan', 'lime', 'orange', 'magenta']):
    qvec = feat[qr * 14 + qc]
    s = (feat @ qvec).reshape(14, 14).numpy()
    s_up = F.interpolate(torch.from_numpy(s)[None, None].float(),
                          size=(S, S), mode='bilinear', align_corners=False).squeeze().numpy()
    ax.imshow(rgb); ax.imshow(s_up, cmap='magma', alpha=0.55)
    ax.add_patch(Rectangle((qc * 16, qr * 16), 16, 16, ec=color, fc='none', lw=2.5))
    ax.set_title(f'query=({qr},{qc})'); ax.axis('off')

plt.suptitle('Prithvi-EO-2.0 — patch similarity from 4 different query locations', y=1.02)
plt.tight_layout(); plt.show()


## 6. Takeaways

- **MAE pretraining gives you patch-level embeddings for free.** No labels, no fine-tuning. The encoder learned to fill in masked patches by leveraging context, and that same representation now lets us ask *which patches behave alike?* on any new HLS chip.
- **Multispectral + multi-temporal context matters.** Prithvi's similarity is computed in a space that already accounts for 6 bands and 4 seasons. Two patches that look identical in a single RGB frame may diverge here because their NIR / SWIR / temporal trajectories differ — exactly what an EO scientist wants.
- **Granularity is set by the patch size.** 16×16 patches at HLS 30 m/px ⇒ each patch ≈ 480 m on the ground. Want finer? Smaller patch size or higher-resolution input, at quadratic compute cost.
- **Limitations to discuss:**
  - The encoder has no notion of *labels* — similarity is whatever the MAE objective made convenient, not a semantic class.
  - Single-scene similarity is local; for cross-scene retrieval we'll need an **ANN index** over many chips' embeddings (coming in a later notebook).
  - Compare side-by-side with **Notebook 2 (DINOv3)** — same scene, different pretraining recipe (self-distillation, RGB-only). The 'similar' patches won't always agree.
